# 06 - Classificação de tendência diária e semanal

Este notebook testa uma pergunta objetiva: **o próximo período tende a fechar em alta ou em baixa?**

São dois experimentos com XGBoost:

- próximo pregão, usando as features diárias;
- próxima semana, usando as features semanais.

Cada experimento respeita a ordem do tempo, compara o modelo com um baseline majoritário e registra parâmetros, métricas e modelo no MLflow.

Uma tendência estimada não é uma ordem de compra ou venda. Antes de qualquer uso operacional ainda seriam necessários validação em várias janelas, backtest, custos, controle de risco e teste em ambiente simulado.

In [ ]:
%pip install mlflow xgboost scikit-learn

In [ ]:
dbutils.library.restartPython()

## Passo 1 - Bibliotecas e funções de avaliação

Como o target tem duas classes, uso métricas de classificação:

- **accuracy**: percentual total de acertos;
- **balanced accuracy**: média do acerto em alta e baixa, útil se as classes estiverem desequilibradas;
- **precision**: entre as previsões de alta, quantas foram realmente alta;
- **recall**: entre as altas reais, quantas o modelo encontrou;
- **F1**: equilíbrio entre precision e recall;
- **ROC-AUC**: capacidade de ordenar casos de alta acima dos casos de baixa.

O baseline sempre prevê a classe mais comum no treino. O XGBoost só agrega valor se superar essa referência simples.

In [ ]:
import mlflow
import mlflow.xgboost
import numpy as np
import pandas as pd

from mlflow.models import infer_signature
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score
)
from xgboost import XGBClassifier

PARAMETROS = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "eval_metric": "logloss"
}

def calcular_metricas(y_real, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_real, y_pred),
        "precision_alta": precision_score(y_real, y_pred, zero_division=0),
        "recall_alta": recall_score(y_real, y_pred, zero_division=0),
        "f1_alta": f1_score(y_real, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_real, y_prob)
    }

def preparar_X(df, features):
    return pd.get_dummies(
        df[features + ["ticker"]],
        columns=["ticker"],
        dtype=float
    )

## Passo 2 - Função de treino temporal

Os 20% finais das datas ficam no teste. Uma data é retirada entre treino e teste, porque o target olha um período à frente. Isso evita que a última linha do treino use como resposta o primeiro período do teste.

A mesma função será usada nos experimentos diário e semanal, garantindo uma comparação metodológica consistente.

In [ ]:
def treinar_e_avaliar(df, date_col, target_col, features, nome_run):
    base = df.dropna(subset=features + [target_col]).copy()
    base[date_col] = pd.to_datetime(base[date_col])
    base = base.sort_values([date_col, "ticker"]).reset_index(drop=True)

    datas = sorted(base[date_col].unique())
    indice_corte = int(len(datas) * 0.8)
    data_inicio_teste = datas[indice_corte]
    data_purga = datas[indice_corte - 1]

    treino = base[base[date_col] < data_purga].copy()
    teste = base[base[date_col] >= data_inicio_teste].copy()

    X_treino = preparar_X(treino, features)
    X_teste = preparar_X(teste, features).reindex(columns=X_treino.columns, fill_value=0)
    y_treino = treino[target_col].astype(int)
    y_teste = teste[target_col].astype(int)

    modelo = XGBClassifier(**PARAMETROS)
    modelo.fit(X_treino, y_treino)
    y_pred = modelo.predict(X_teste)
    y_prob = modelo.predict_proba(X_teste)[:, 1]
    metricas_modelo = calcular_metricas(y_teste, y_pred, y_prob)

    classe_majoritaria = int(y_treino.mode().iloc[0])
    y_pred_baseline = np.full(len(y_teste), classe_majoritaria)
    y_prob_baseline = np.full(len(y_teste), y_treino.mean())
    metricas_baseline = calcular_metricas(y_teste, y_pred_baseline, y_prob_baseline)

    with mlflow.start_run(run_name=nome_run):
        mlflow.log_params(PARAMETROS)
        mlflow.log_param("target", target_col)
        mlflow.log_param("data_inicio_teste", str(pd.Timestamp(data_inicio_teste).date()))
        for nome, valor in metricas_modelo.items():
            mlflow.log_metric(nome, valor)
        for nome, valor in metricas_baseline.items():
            mlflow.log_metric(f"baseline_{nome}", valor)

        assinatura = infer_signature(X_treino, modelo.predict_proba(X_treino)[:, 1])
        mlflow.xgboost.log_model(
            modelo,
            name="modelo",
            input_example=X_treino.head(5),
            signature=assinatura
        )

    matriz = confusion_matrix(y_teste, y_pred)
    print(f"{nome_run}: treino={len(treino)} | teste={len(teste)}")
    print(f"Início do teste: {pd.Timestamp(data_inicio_teste).date()}")
    print("Matriz de confusão [[baixa correta, alta como baixa], [baixa como alta, alta correta]]:")
    print(matriz)

    return {
        "run": nome_run,
        "modelo": modelo,
        "features": features,
        "colunas_X": X_treino.columns,
        "metricas_modelo": metricas_modelo,
        "metricas_baseline": metricas_baseline,
        "matriz_confusao": matriz
    }

## Passo 3 - Tendência do próximo pregão

O modelo diário tenta classificar se o fechamento do próximo pregão será maior que o atual. Todas as features foram calculadas no notebook 05 usando somente o fechamento atual e o passado.

In [ ]:
df_diario = spark.table("b3_pipeline.gold_tendencia_features_diario").toPandas()

FEATURES_DIARIAS = [
    "daily_return_pct",
    "distancia_media_5d_pct", "distancia_media_20d_pct",
    "momentum_5d_pct", "momentum_20d_pct",
    "volatilidade_5d", "volatilidade_20d",
    "retorno_lag1", "retorno_lag2", "retorno_lag5"
]

resultado_diario = treinar_e_avaliar(
    df=df_diario,
    date_col="date",
    target_col="target_tendencia_prox_dia",
    features=FEATURES_DIARIAS,
    nome_run="xgboost_tendencia_diaria"
)

## Passo 4 - Tendência da próxima semana

O segundo modelo usa os fechamentos semanais e tenta classificar se a próxima semana fechará acima da atual. As métricas não devem ser comparadas diretamente com as diárias sem considerar que os horizontes e o número de observações são diferentes.

In [ ]:
df_semanal = spark.table("b3_pipeline.gold_tendencia_features_semanal").toPandas()

FEATURES_SEMANAIS = [
    "weekly_return_pct",
    "distancia_media_4s_pct", "distancia_media_12s_pct",
    "momentum_4s_pct", "momentum_12s_pct",
    "volatilidade_4s", "volatilidade_12s",
    "retorno_lag1s", "retorno_lag2s", "retorno_lag4s"
]

resultado_semanal = treinar_e_avaliar(
    df=df_semanal,
    date_col="week_start",
    target_col="target_tendencia_prox_semana",
    features=FEATURES_SEMANAIS,
    nome_run="xgboost_tendencia_semanal"
)

## Passo 5 - Comparação com os baselines

A tabela abaixo não procura apenas a maior acurácia. O ponto principal é verificar se cada modelo supera o seu próprio baseline em balanced accuracy, F1 e ROC-AUC.

In [ ]:
linhas = []
for horizonte, resultado in [("diario", resultado_diario), ("semanal", resultado_semanal)]:
    linhas.append({"horizonte": horizonte, "abordagem": "xgboost", **resultado["metricas_modelo"]})
    linhas.append({"horizonte": horizonte, "abordagem": "baseline", **resultado["metricas_baseline"]})

comparacao = pd.DataFrame(linhas)
display(comparacao)

## Passo 6 - Estimativas mais recentes por ticker

Depois da avaliação, treino uma versão final com todo o histórico rotulado e aplico nas linhas mais recentes. A probabilidade é transformada em três leituras:

- `alta` quando `prob_alta >= 0.55`;
- `baixa` quando `prob_alta <= 0.45`;
- `indefinida` na faixa intermediária.

A faixa indefinida evita transformar uma probabilidade próxima de 50% em uma certeza artificial.

In [ ]:
def gerar_sinais_recentes(df, date_col, target_col, features, horizonte):
    base_treino = df.dropna(subset=features + [target_col]).copy()
    base_treino[date_col] = pd.to_datetime(base_treino[date_col])

    recentes = df.dropna(subset=features).copy()
    recentes[date_col] = pd.to_datetime(recentes[date_col])
    recentes = recentes.sort_values(date_col).groupby("ticker", as_index=False).tail(1).copy()

    X_completo = preparar_X(base_treino, features)
    X_recentes = preparar_X(recentes, features).reindex(columns=X_completo.columns, fill_value=0)
    y_completo = base_treino[target_col].astype(int)

    modelo_final = XGBClassifier(**PARAMETROS)
    modelo_final.fit(X_completo, y_completo)
    prob_alta = modelo_final.predict_proba(X_recentes)[:, 1]

    sinais = recentes[["ticker", date_col]].copy()
    sinais[date_col] = sinais[date_col].dt.date
    sinais["horizonte"] = horizonte
    sinais["prob_alta"] = prob_alta
    sinais["prob_baixa"] = 1 - prob_alta
    sinais["tendencia_estimada"] = np.where(
        prob_alta >= 0.55, "alta",
        np.where(prob_alta <= 0.45, "baixa", "indefinida")
    )
    return sinais

sinais_diarios = gerar_sinais_recentes(
    df_diario, "date", "target_tendencia_prox_dia", FEATURES_DIARIAS, "proximo_pregao"
)
sinais_semanais = gerar_sinais_recentes(
    df_semanal, "week_start", "target_tendencia_prox_semana", FEATURES_SEMANAIS, "proxima_semana"
)

display(sinais_diarios.sort_values("prob_alta", ascending=False))
display(sinais_semanais.sort_values("prob_alta", ascending=False))

## Passo 7 - Salvando os sinais para a camada GenAI

As duas tabelas são salvas na Gold. Em uma evolução futura, esses sinais poderão ser combinados com o experimento de sentimento do notebook 07, sem alterar o treinamento dos classificadores.

In [ ]:
spark.createDataFrame(sinais_diarios).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.gold_sinais_tendencia_diaria")

spark.createDataFrame(sinais_semanais).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.gold_sinais_tendencia_semanal")

print("Sinais diário e semanal salvos com sucesso!")

## Como interpretar

O resultado relevante é a comparação contra o baseline no período de teste. As estimativas mais recentes servem apenas para demonstrar a saída do pipeline.

Antes de relacionar `alta` a uma posição comprada ou `baixa` a uma posição vendida, ainda faltam validação walk-forward, backtest com custos, drawdown, regras de entrada e saída e teste em ambiente simulado.